# Population Regression from Images (PyTorch)

Train a CNN to predict a population number from each image. The label is parsed
from the filename: the integer between the **last underscore** and `.jpg`
(e.g. `city_region_12345.jpg` -> `12345`).

Run the cells top to bottom. Edit the **Config** cell to change anything.

> **Running on CPU.** The defaults below (small CNN, no pretrained weights,
> balanced + downsampled data, 8 epochs) are tuned to train in minutes on CPU.
> If you later get a GPU working, see the optional note in section 1a.

## 1a. (Optional) GPU later

This notebook runs on CPU as configured. If you ever want to revisit GPU
acceleration on your AMD card, it requires the ROCm build of PyTorch in a
**Python 3.12** environment (the wheels are `cp312`-only; that's what caused the
earlier *circular import* error on Python 3.14). Not needed for now -- the CPU
defaults below are tuned to be fast enough. The device-selection cell still
auto-detects a GPU if one is ever available, so nothing here needs to change.

## 1. Config
Everything tunable lives here.

In [1]:
import os, math, time, random
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm.auto import tqdm   # progress bars (pip install tqdm)

# ---------------- EDIT THESE ----------------
DATA_DIR   = r"..\\Data"   # folder of .jpg files
IMAGE_SIZE = 200             # CPU speed/detail tradeoff. 200=native (~1hr+), 96=fastest (~20min), 128~35min for 8 epochs
VAL_SPLIT  = 0.15
TEST_SPLIT = 0.10
BATCH_SIZE = 64              # raise if GPU memory allows; lower if you hit OOM
EPOCHS     = 8               # fewer epochs (was 30)
LR         = 5e-4
WEIGHT_DECAY = 1e-4
BACKBONE   = "simple_cnn"    # CPU-friendly small CNN. Use "resnet18" if you get a GPU.
PRETRAINED = False           # pretrained only matters for the resnet backbones
DROPOUT    = 0.2
NUM_WORKERS = 0              # 0 is safest on Windows; try 4 on Linux for speed
EARLY_STOP_PATIENCE = 4
GRAD_CLIP  = 1.0
SEED       = 42

# Balance the data so there are as many zero-population images as non-zero ones
# (applied to train AND test). See section 3.
BALANCE_ZERO_NONZERO = True

# Target normalization. Populations here are a small bounded range (0-200, mostly
# 0), so a log transform is NOT appropriate -- it would distort this range. We
# leave LOG_TARGET off and just standardize (mean 0, std 1, fit on train only),
# which keeps the optimizer well-conditioned. Predictions are inverted back to
# real counts for all reported metrics.
LOG_TARGET = False
STD_TARGET = True

OUT_DIR    = "run"
# --------------------------------------------

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 1b. Device check

Reports the device in use. CPU is expected here and is fine with the small-CNN
defaults. It will automatically use a GPU if one is ever available.

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")          # ROCm presents AMD GPUs via the CUDA API
    name = torch.cuda.get_device_name(0)
    print(f"device: cuda  ->  {name}")
    print("HIP/ROCm version:", getattr(torch.version, "hip", None))
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("device: mps (Apple Silicon)")
else:
    device = torch.device("cpu")
    print("device: cpu  (expected; small-CNN defaults are tuned for this)")

# cudnn.benchmark only matters on GPU; harmless on CPU
torch.backends.cudnn.benchmark = True
print("CPU threads available:", torch.get_num_threads())

device: cpu  (expected; small-CNN defaults are tuned for this)
CPU threads available: 6


## 2. Read filenames -> population labels

`extract_population` is the **only** part tied to your naming scheme. To change
the label source later, edit just this function -- return an `int`, or `None` to
skip a file.

In [3]:
def extract_population(filename: str):
    """Integer between the last '_' and '.jpg', or None if not parseable."""
    if not filename.lower().endswith(".jpg"):
        return None
    stem = os.path.splitext(filename)[0]
    token = stem.split("_")[-1]
    try:
        return int(token)
    except ValueError:
        return None

paths, pops = [], []
skipped = 0
for name in sorted(os.listdir(DATA_DIR)):
    if not name.lower().endswith(".jpg"):
        continue
    p = extract_population(name)
    if p is None:
        skipped += 1
        continue
    paths.append(os.path.join(DATA_DIR, name))
    pops.append(p)

paths = np.array(paths); pops = np.array(pops)
assert len(paths), f"No usable .jpg files found in {DATA_DIR!r}"
if skipped:
    print(f"skipped {skipped} file(s) with unparseable population")
n_zero = int((pops == 0).sum()); n_nonzero = int((pops != 0).sum())
print(f"found {len(paths)} labeled images  |  zero-pop: {n_zero}   non-zero: {n_nonzero}")

found 9438 labeled images  |  zero-pop: 8002   non-zero: 1436


## 3. Balance, then split

**Balancing (point 2):** we split the data into zero-population and
non-zero-population groups and randomly downsample the larger group so the two
are equal in size. Because we balance *first* and split *each group* with the
same ratios, the resulting train, val, and test sets are each ~50/50 zero vs.
non-zero. (Downsampling means some images of the majority class are not used --
that's the cost of a balanced set.)

In [4]:
rng = np.random.default_rng(SEED)
zero_idx    = np.where(pops == 0)[0]
nonzero_idx = np.where(pops != 0)[0]

if BALANCE_ZERO_NONZERO:
    k = min(len(zero_idx), len(nonzero_idx))
    if k == 0:
        raise ValueError("Cannot balance: one of the groups is empty.")
    zero_sel    = rng.choice(zero_idx,    size=k, replace=False)
    nonzero_sel = rng.choice(nonzero_idx, size=k, replace=False)
    print(f"balancing: using {k} zero + {k} non-zero = {2*k} images "
          f"(dropped {len(paths) - 2*k} from the majority class)")
else:
    zero_sel, nonzero_sel = zero_idx, nonzero_idx

def split_group(indices):
    """Shuffle one group and cut it into train/val/test by the global ratios."""
    indices = rng.permutation(indices)
    n = len(indices)
    n_test = int(round(n * TEST_SPLIT))
    n_val  = int(round(n * VAL_SPLIT))
    return (indices[n_test + n_val:],        # train
            indices[n_test:n_test + n_val],  # val
            indices[:n_test])                # test

# Split each class separately, then concatenate -> every split stays ~balanced
ztr, zva, zte = split_group(zero_sel)
ntr, nva, nte = split_group(nonzero_sel)
train_idx = rng.permutation(np.concatenate([ztr, ntr]))
val_idx   = rng.permutation(np.concatenate([zva, nva]))
test_idx  = rng.permutation(np.concatenate([zte, nte]))

def take(ix):
    return list(paths[ix]), [int(v) for v in pops[ix]]

train_paths, train_pops = take(train_idx)
val_paths,   val_pops   = take(val_idx)
test_paths,  test_pops  = take(test_idx)

def frac_zero(p):
    return (np.array(p) == 0).mean() if len(p) else float("nan")
print(f"train={len(train_paths)} ({frac_zero(train_pops):.0%} zero)  "
      f"val={len(val_paths)} ({frac_zero(val_pops):.0%} zero)  "
      f"test={len(test_paths)} ({frac_zero(test_pops):.0%} zero)")

balancing: using 1436 zero + 1436 non-zero = 2872 images (dropped 6566 from the majority class)
train=2154 (50% zero)  val=430 (50% zero)  test=288 (50% zero)


## 4. Target normalization (fit on train only)

In [5]:
_t = np.asarray(train_pops, dtype=np.float64)
if LOG_TARGET:
    _t = np.log1p(_t)
TARGET_MEAN = float(_t.mean()) if STD_TARGET else 0.0
TARGET_STD  = (float(_t.std()) or 1.0) if STD_TARGET else 1.0

def encode_target(value: float) -> float:
    x = math.log1p(value) if LOG_TARGET else float(value)
    if STD_TARGET:
        x = (x - TARGET_MEAN) / TARGET_STD
    return x

def decode_target(x):
    x = x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x, dtype=np.float64)
    if STD_TARGET:
        x = x * TARGET_STD + TARGET_MEAN
    if LOG_TARGET:
        x = np.expm1(x)
    return x

print(f"target mean={TARGET_MEAN:.4f}  std={TARGET_STD:.4f}")

target mean=7.9717  std=17.9201


## 5. Dataset & DataLoaders
Augmentation on the train split only.

In [6]:
train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class PopulationDataset(Dataset):
    def __init__(self, paths, pops, tf):
        self.paths, self.pops, self.tf = paths, pops, tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        try:
            with Image.open(self.paths[i]) as img:
                image = self.tf(img.convert("RGB"))
        except (OSError, ValueError) as e:
            print(f"warn: failed to load {self.paths[i]}: {e}; using blank")
            image = self.tf(Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE)))
        target = torch.tensor(encode_target(self.pops[i]), dtype=torch.float32)
        return image, target

train_ds = PopulationDataset(train_paths, train_pops, train_tf)
val_ds   = PopulationDataset(val_paths,   val_pops,   eval_tf)
test_ds  = PopulationDataset(test_paths,  test_pops,  eval_tf)

_loader = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
               pin_memory=(device.type == "cuda"))
train_ld = DataLoader(train_ds, shuffle=True, drop_last=True, **_loader)
val_ld   = DataLoader(val_ds,   shuffle=False, **_loader)
test_ld  = DataLoader(test_ds,  shuffle=False, **_loader)
print(f"batches/epoch: train={len(train_ld)}  val={len(val_ld)}  test={len(test_ld)}")

batches/epoch: train=33  val=7  test=5


## 6. Model

In [7]:
class SimpleCNN(nn.Module):
    def __init__(self, dropout=0.2):
        super().__init__()
        def block(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3,32), block(32,64), block(64,128), block(128,256))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(dropout), nn.Linear(256,128), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(128,1))
    def forward(self, x):
        return self.head(self.features(x)).squeeze(1)

def build_model():
    if BACKBONE == "simple_cnn":
        return SimpleCNN(DROPOUT)
    ctor, weights = {
        "resnet18": (models.resnet18, models.ResNet18_Weights.IMAGENET1K_V1),
        "resnet34": (models.resnet34, models.ResNet34_Weights.IMAGENET1K_V1),
        "resnet50": (models.resnet50, models.ResNet50_Weights.IMAGENET1K_V2),
    }[BACKBONE]
    net = ctor(weights=weights if PRETRAINED else None)
    net.fc = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(net.fc.in_features, 1))
    return net

model = build_model().to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{BACKBONE}: {n_params:,} trainable params")

simple_cnn: 1,207,201 trainable params


## 7. Loss, optimizer, scheduler, metrics

In [8]:
loss_fn = nn.SmoothL1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    err = y_pred - y_true
    mae  = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err**2)))
    ss_res = float(np.sum(err**2))
    ss_tot = float(np.sum((y_true - y_true.mean())**2)) or 1.0
    r2 = 1.0 - ss_res/ss_tot
    nz = y_true != 0
    mape = float(np.mean(np.abs(err[nz]/y_true[nz]))*100) if nz.any() else float("nan")
    return {"mae": mae, "rmse": rmse, "r2": r2, "mape": mape}

## 8. Train & evaluate loops

The training loop uses a **`tqdm` progress bar per epoch** showing batch progress
within the epoch, live loss, and iterations/sec (point 4). An outer bar tracks
overall epoch progress.

In [9]:
def run_epoch(loader, train: bool, epoch: int, n_epochs: int):
    model.train() if train else model.eval()
    total_loss, enc_true, enc_pred = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    desc = f"Epoch {epoch}/{n_epochs} " + ("train" if train else "val  ")
    bar = tqdm(loader, desc=desc, leave=False, unit="batch")
    with ctx:
        for step, (images, targets) in enumerate(bar, 1):
            images  = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            preds = model(images)
            if preds.ndim > 1:
                preds = preds.squeeze(1)
            loss = loss_fn(preds, targets)
            if train:
                optimizer.zero_grad(); loss.backward()
                if GRAD_CLIP > 0:
                    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
            total_loss += loss.item()
            enc_true.append(targets.detach().cpu().numpy())
            enc_pred.append(preds.detach().cpu().numpy())
            bar.set_postfix(loss=f"{total_loss/step:.4f}")
    bar.close()
    enc_true = np.concatenate(enc_true); enc_pred = np.concatenate(enc_pred)
    metrics = regression_metrics(decode_target(enc_true), decode_target(enc_pred))
    return total_loss / max(len(loader), 1), metrics

## 9. Train

In [10]:
best_val = float("inf"); patience = 0
ckpt_path = os.path.join(OUT_DIR, "best_model.pt")

epoch_bar = tqdm(range(1, EPOCHS + 1), desc="Total", unit="epoch")
for epoch in epoch_bar:
    train_loss, _    = run_epoch(train_ld, True,  epoch, EPOCHS)
    val_loss, val_m  = run_epoch(val_ld,   False, epoch, EPOCHS)
    scheduler.step()
    epoch_bar.set_postfix(train=f"{train_loss:.3f}", val=f"{val_loss:.3f}",
                          MAE=f"{val_m['mae']:.0f}", R2=f"{val_m['r2']:.2f}")
    tqdm.write(f"epoch {epoch:2d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  "
               f"MAE={val_m['mae']:.1f}  RMSE={val_m['rmse']:.1f}  R2={val_m['r2']:.3f}")
    if val_loss < best_val:
        best_val = val_loss; patience = 0
        torch.save({"model_state": model.state_dict(),
                    "target": {"mean": TARGET_MEAN, "std": TARGET_STD,
                               "log": LOG_TARGET, "std_on": STD_TARGET},
                    "backbone": BACKBONE, "epoch": epoch}, ckpt_path)
        tqdm.write(f"  -> new best, saved {ckpt_path}")
    else:
        patience += 1
        if patience >= EARLY_STOP_PATIENCE:
            tqdm.write(f"early stop after {patience} epochs without improvement")
            break
epoch_bar.close()

Total:   0%|          | 0/8 [00:00<?, ?epoch/s]

Epoch 1/8 train:   0%|          | 0/33 [00:00<?, ?batch/s]

Epoch 1/8 val  :   0%|          | 0/7 [00:00<?, ?batch/s]

epoch  1/8  train=0.1848  val=0.2239  MAE=6.8  RMSE=16.1  R2=-0.028
  -> new best, saved run\best_model.pt


Epoch 2/8 train:   0%|          | 0/33 [00:00<?, ?batch/s]

...

Epoch 7/8 val  :   0%|          | 0/7 [00:00<?, ?batch/s]

epoch  7/8  train=0.1397  val=0.1050  MAE=4.2  RMSE=10.6  R2=0.551
  -> new best, saved run\best_model.pt


Epoch 8/8 train:   0%|          | 0/33 [00:00<?, ?batch/s]

Epoch 8/8 val  :   0%|          | 0/7 [00:00<?, ?batch/s]

epoch  8/8  train=0.1323  val=0.1023  MAE=4.1  RMSE=10.5  R2=0.560
  -> new best, saved run\best_model.pt


## 10. Final test evaluation (best checkpoint)

In [11]:
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
test_loss, test_m = run_epoch(test_ld, False, ckpt["epoch"], EPOCHS)
print(f"TEST  loss={test_loss:.4f}  MAE={test_m['mae']:.1f}  RMSE={test_m['rmse']:.1f}  "
      f"R2={test_m['r2']:.3f}  MAPE={test_m['mape']:.1f}%")

Epoch 8/8 val  :   0%|          | 0/5 [00:00<?, ?batch/s]

TEST  loss=0.0990  MAE=4.1  RMSE=10.9  R2=0.553  MAPE=527078132.2%


In [13]:
#Show distribution of actual vs predicted populations on the test set
import matplotlib.pyplot as plt
model.eval()
all_true, all_pred = [], []

with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device)
        targets = targets.to(device)
        outputs = model(images)
        all_true.extend(targets.cpu().numpy())
        all_pred.extend(outputs.cpu().numpy())
plt.figure(figsize=(8, 6))
plt.scatter(all_true, all_pred, alpha=0.5)
plt.plot([0, max(all_true)], [0, max(all_true)], 'r--')  # y=x line
plt.xlabel('Actual Population')
plt.ylabel('Predicted Population')
plt.title('Actual vs Predicted Population on Test Set')
plt.grid()
plt.show()



NameError: name 'test_loader' is not defined

## 11. Predict on new images

In [12]:
def predict(image_paths):
    if isinstance(image_paths, str):
        if os.path.isdir(image_paths):
            image_paths = [os.path.join(image_paths, f) for f in sorted(os.listdir(image_paths))
                           if f.lower().endswith(".jpg")]
        else:
            image_paths = [image_paths]
    model.eval()
    out = []
    with torch.no_grad():
        for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="predict", unit="batch"):
            chunk = image_paths[i:i+BATCH_SIZE]
            batch, names = [], []
            for f in chunk:
                try:
                    with Image.open(f) as img:
                        batch.append(eval_tf(img.convert("RGB"))); names.append(f)
                except Exception as e:
                    print(f"skip {f}: {e}")
            if not batch:
                continue
            preds = model(torch.stack(batch).to(device))
            if preds.ndim > 1:
                preds = preds.squeeze(1)
            for n, r in zip(names, np.atleast_1d(decode_target(preds))):
                out.append((os.path.basename(n), float(r)))
    return out

# Example:
# for name, pop in predict(r"..\\NewImages"):
#     print(f"{name}\t{pop:,.0f}")